In [3]:
# -*- coding: utf-8 -*-
"""
Created on Wed Jun 14 16:40:19 2023

@author: Sabyasachi
"""

# -*- coding: utf-8 -*-
"""
Created on Tue Jul 23 2019

@author: arazza,sgonzalez
"""

'''
Demo script as Prospector tutorial at CRISP meeting 2019.

The demo is devised to run Prospector to fit spectral data with the same sampling and
istrumental broadening as MUSE, but covering a bluer wavelength range.

Basic fitting ingredients are parsed as command line arguments, with some extra
parameters added to control for instance the noise level to be added to the mock test.

The main Prospector build methods are implemeted:
- build_model
  Priors for free parameters to be fitted and initial guesses for both free and fixed parameters are defined
- build_obs
  A FSPS model is used as mock spectrum to be fitted. Noise at different levels can be added.
- build_sps
  To instantiate the Stellar Population Synthesis (sps) object
'''
from matplotlib import pyplot as plt

import time, sys, os, pdb
from copy import deepcopy
import numpy as np
from sedpy.observate import load_filters
import sedpy
from prospect import prospect_args
from prospect.fitting import fit_model
from prospect.models import transforms

## These built-in prospector fcts did not save properly when only optimizing:
from prospect.io import write_results as writer
from prospect.io import read_results as reader
## So we use:
#from h5file import write_h5file,read_h5file
from mass_to_light_compute import mass_prior
#from CRISP_15d_plot_photo_vor import plot_results,plot_spectrum
#from ewcalculation import ewspec,plotspectrum
from prospect.models.sedmodel import SpecModel

# --------------
# Minimization function (instead of default)
# --------------

def chivecfn(theta,**kwargs):
    """A version of lnprobfn that returns the simple uncertainty
    normalized residual instead of the log-posterior, for use with
    least-squares optimization methods like Levenburg-Marquardt.

    It's important to note that the returned chi vector does not
    include the prior probability.
    """
    from prospect.likelihood import chi_spec, chi_phot
    lnp_prior = model.prior_product(theta)
    if not np.isfinite(lnp_prior):
        return np.zeros(model.ndim) - np.infty

    # Generate mean model
    try:
        spec, phot, x = model.mean_model(theta, obs, sps=sps)
    except(ValueError):
        return np.zeros(model.ndim) - np.infty

    chispec = chi_spec(spec, obs)
    chiphot = chi_phot(phot, obs)
    return np.concatenate([chispec, chiphot])

def lnprobfn(theta, model=None, obs=None, sps=None,
             nested=False, verbose=False,**kwargs):
    #
    """
    Given a parameter vector, a model, a dictionary of observational
    data, and an sps object, return the ln of the posterior.
    """
    from prospect.likelihood import lnlike_spec, lnlike_phot, write_log
    # Calculate prior probability and exit if not within prior
    # Also if doing nested sampling, do not include the basic priors,
    # since the drawing method includes the prior probability
    lnp_prior = model.prior_product(theta, nested=nested)
    if not np.isfinite(lnp_prior):
        return -np.infty

    # Generate "mean" model
    spec, phot, mfrac = model.mean_model(theta, obs, sps=sps)

    # Calculate likelihoods
    lnp_spec = lnlike_spec(spec, obs=obs)
    lnp_phot = lnlike_phot(phot, obs=obs)

    return lnp_prior + lnp_phot + lnp_spec

# --------------
# Model Definition
# --------------

def build_model(add_duste=False, add_neb=False,massprior=None,**kwargs):
    """Instantiate and return a ProspectorParams model subclass.

    :param zred: (optional, default: 0.0)
        The redshift of the model

    :param add_neb: (optional, default: False)
        If True, turn on nebular emission and add relevant parameters to the
        model.
    """
    from prospect.models.templates import TemplateLibrary
    from prospect.models import priors, sedmodel
    from prospect.sources.constants import cosmo

    # --- Get a basic delay-tau SFH parameter set. ---
    # This has 5 free parameters:
    #   "mass", "logzsol", "dust2", "tage", "tau"
    # And two fixed parameters
    #   "zred"=0.1, "sfh"=4
    # See the python-FSPS documentation for details about most of these
    # parameters.  Also, look at `TemplateLibrary.describe("parameteric")` to
    # view the parameters, their initial values, and the priors in detail
    model_params = TemplateLibrary["continuity_sfh"]
    #model_params = TemplateLibrary["alpha"]
    #model_params.update(TemplateLibrary["optimize_speccal"])

    # --- Adjust the basic model ----
    # Add spectral smoothing (fixed to MUSE instrumental)
    model_params.update(TemplateLibrary["spectral_smoothing"])
    # Add dust emission parameters (fixed)
    #if add_duste:
   # model_params.update(TemplateLibrary["dust_emission"])
    # Add nebular emission parameters and turn nebular emission on
    if add_neb:
        model_params.update(TemplateLibrary["nebular"])
    zred=0.005294

    # --- Redshift  ---
    # Switch to Milky Way extinction law parameterized by Cardelli et al. 1989, with variable UV bump
    # strength; see variables mwr and uvb below.
    model_params["zred"] = {'N': 1, 'isfree': True,
                            'init':0.005294, 'prior': priors.TopHat(mini=zred-0.01, maxi=zred+0.01)}#0.03334  0.024272  0.005158  0.005294



    # --- Complexify dust attenuation ---
    # Switch to Milky Way extinction law parameterized by Cardelli et al. 1989, with variable UV bump
    # strength; see variables mwr and uvb below.
    model_params["dust_type"] = {'N': 1, 'isfree': False,
                                 'init': 4, 'prior': None}

    # mwr is the ratio of total to selective absorption which characterizes the MW extinction curve:
    # R ≡ AV /E(B − V ). Only used when dust type=1. Default value is 3.1.
    model_params["dust_index"] = {'N': 1, 'isfree': True,
                                 'init':-0.6 ,#'init_disp' : 1.0,#,'disp_floor':0.5,
                                   'prior': priors.TopHat(mini=-2.2, maxi=0.4)}

#    model_params["gas_logu"] = {'N': 1, 'isfree': False,
#                                 'init': -2.5, 'prior':priors.TopHat(mini=-3.00, maxi=-1.0) }#priors.TopHat(mini=1.5, maxi=4.5)}
    # uvb Parameter characterizing the strength of the 2175 Angs extinction feature with respect to
    # the standard Cardelli et al. determination for the MW. Only used when dust type=1. Default value is 1.0.
    #model_params["uvb"] = {'N': 1, 'isfree': False,
     #                            'init': 1.0, 'prior': None}
#    model_params["logmass"] = {'N': 1, 'isfree': False,
#                                 'init': 6.5}#, 'prior': priors.LogUniform(mini=1e4,maxi=1e14)}
    model_params['sigma_smooth'] = {'N': 1, 'isfree': True,
                 'init': 120.0, 'units': 'km/s', 'init_disp' : 200.0,
                 'prior': priors.TopHat(mini=70, maxi=200)}
    model_params["logzsol"] = {"N": 1, "isfree": True,
           "init": -0.9,'init_disp' : 5.0,
           "units": r"$\log (Z/Z_\odot)$",
           "prior": priors.TopHat(mini=-2.0, maxi=0.19)}
    # --- Set wavelength space for sigma_smooth ---
   # model_params["smoothtype"]["init"]=None
   # model_params["smoothtype"]["init"]="lambda" # spectrum in wavelength space
    #model_params["duste_qpah"]["isfree"] = True   # if you want to fit for the IR SED shape
    #model_params["duste_umin"]["isfree"] = True   # if you want to fit for the IR SED shape
    # --- Set initial values ---
    #input mock: 1e7,-0.5,0.5,1.0,1.0,1.126,3.1
    #mass,logzsol,dust2,tage,tau,sig,rv,zred = 1e7,0.5,0.5,5.0,5.0,0.0,3.1,0.03334#0.03334#,0.024272

    mass,logzsol,dust2,rv,zred =  8.4,-0.5,0.5,3.1,0.005294#0.005294#0.03334#,0.024272
#    #mass,logzsol,dust2,tage,tau,sig,rv,zred = 1.38e7,0.0,1.63,0.404,2.97,10.0,3.1,0.033
    # 7
#    fr = 0.8
    model_params["zred"]["init"] = zred
    model_params["logmass"]["init"] = mass
    #model_params["logzsol"]["init"] = logzsol
    model_params["dust2"]["init"] = dust2
    #model_params["tage"]["init"] = tage
    #model_params["tau"]["init"] = tau
    #model_params["sigma_smooth"]["init"] = sig
    #model_params["mwr"]["init"] = rv
    nbins_sfh=7


    model_params["agebins"]    = {'N': nbins_sfh, 'isfree': False,
                                     'init': [[ 0. , 7.4772 ] ,[ 7.4772 ,8. ] ,[ 8. ,8.49936142] ,[ 8.49936142 ,8.99872284] ,[ 8.99872284, 9.49808425], [ 9.49808425, 9.99744567] ,[ 9.99744567, 10.06802675]],
                                     'units': 'log(yr)'}


#    'N': 3, 'isfree': False,
#                                     'init': [[0.0, 8.0], [8.0, 9.0], [9.0, 10.0]],
#                                     'units': 'log(yr)'}
    #model_params['agebins']['init'] = agebins.T
    #model_params['mass']['N'] = nbins_sfh

    #model_params['mass']={'N': nbins_sfh, 'isfree': False, 'init': 1e6, 'units': r'M$_\odot$','depends_on': transforms.logsfr_ratios_to_masses}
    model_params['agebins']['N'] = nbins_sfh


    model_params['logsfr_ratios']['N'] = nbins_sfh-1
    model_params['logsfr_ratios']['init'] = np.full(nbins_sfh-1,0.0) # constant SFH
    model_params['logsfr_ratios']['prior'] = priors.StudentT(mean=np.full(nbins_sfh-1,0.0),
                                                                  scale=np.full(nbins_sfh-1,0.3),
                                                                  df=np.full(nbins_sfh-1,2))
    model_params['logsfr_ratios']['init_disp']=5.0
    # --- Set priors for free parameters --- (open)
    def zred_to_agebins(zred=None, nbins_sfh=7, **extras):
       tuniv = np.squeeze(cosmo.age(zred).to("yr").value)
       ncomp = np.squeeze(nbins_sfh)
       tbinmax = (tuniv*0.9)
       agelims = [0.0, 7.4772] + np.linspace(8.0, np.log10(tbinmax), ncomp-2).tolist() + [np.log10(tuniv)]
       agebins = np.array([agelims[:-1], agelims[1:]])
       return agebins.T
    def logmass_to_masses(logmass=None, logsfr_ratios=None, zred=None, **extras):
        agebins = zred_to_agebins(zred=zred, **extras)
        logsfr_ratios = np.clip(logsfr_ratios, -10, 10)  # numerical issues...
        nbins = agebins.shape[0]
        sratios = 10**logsfr_ratios
        dt = (10**agebins[:, 1] - 10**agebins[:, 0])
        coeffs = np.array([(1./np.prod(sratios[:i])) * (np.prod(dt[1:i+1]) / np.prod(dt[:i])) for i in range(nbins)])
        m1 = (10**logmass) / coeffs.sum()
        return m1 * coeffs
    model_params['agebins']['depends_on'] = zred_to_agebins
    model_params['mass']['depends_on'] = logmass_to_masses
#modifedtest
   # model_params["zred"]["prior"] = priors.TopHat(mini=0.004, maxi=0.006)
    #model_params["logzsol"]["prior"] = priors.TopHat(mini=-2.00, maxi=0.19)
    model_params["logmass"]["prior"] = priors.TopHat(mini=6, maxi=12.0)
    model_params["dust2"]["prior"] = priors.ClippedNormal(mini=0.0, maxi=3.0, mean=0.3, sigma=1)
    #model_params["tage"]["prior"] = priors.TopHat(mini=0.001,maxi=13.8)
   # model_params["tau"]["prior"] = priors.LogUniform(mini=0.1,maxi=30.0)
    #model_params["sigma_smooth"]["prior"] = priors.TopHat(mini=0.0, maxi=10.0)
   # model_params["mwr"]["prior"] = priors.TopHat(mini=0.5, maxi=5.5)

    #mass prior
#    if massprior is not None:
#        model_params["mass"]["init"] = massprior[1]
#        model_params["mass"]["prior"] = priors.LogUniform(mini=massprior[0], maxi=massprior[2])

  #   --- Set dispersions for emcee ---
    model_params["logmass"]["init_disp"] = 5.0

    #model_params["mass"]["disp_floor"] = 1e6
    model_params["dust2"]["init_disp"] = 5.0
    #model_params["dust2"]["disp_floor"] = 0.2

    #model=sedmodel.SedModel(model_params)
    model=SpecModel(model_params)
    return model

# -------------------
# Get mask
# ------------------
def get_mask(wave,maskfile= 'mask.dat',nomask=False,**kwargs):

    mask = np.array(np.ones(wave.shape), bool)
    if nomask: return mask

    #read mask
#    file = open(maskfile,'r')
#    waverange = np.asarray([line.split()[0:2] for line in file],dtype=list)#.astype(int)#float tirar astype
#    file.close()
    waverange=np.loadtxt(maskfile, usecols=(0,1))
    waverange= waverange*(1.0+ 0.005294)
    #waverange=file.flatten()

    #assign mask
    for wmin,wmax in waverange:
        mask[(wave > wmin) & (wave < wmax)] = False
    return mask



# -------------------
# Get INLA prior -> NOT IMPLEMENTED
# -------------------
def inla_prior():

    input = np.loadtxt("input.txt", dtype=object, delimiter=';',skiprows=1)


# --------------------------
# Voronoi binning (see vorbin code)
# --------------------------
def voronoi_bin(cube,snr,**kwargs):


    odir = kwargs['outdir']
    gfile = kwargs['genfile']
    perc = kwargs['perc']
    ## check if file exists!
    if os.path.isfile(odir+gfile+'_voronoi_2d_binning_output.npy'):
        binNum,xBar,yBar,flux,eflux = np.load(odir+gfile+'_voronoi_2dbinning_output.npy')

    from vorbin.voronoi_2d_binning import voronoi_2d_binning

    # find 1d arrays
    (nwave,ny,nx) = np.shape(cube['flux'])
    (nwave_p,ny_p,nx_p) = np.shape(cube['photo_flux'])

    nsz = ny*nx

    xx,yy = np.meshgrid(np.arange(0,nx),np.arange(0,ny))
    x,y = np.zeros(nsz,dtype=float),np.zeros(nsz,dtype=float)
    flux,eflux = np.zeros((nwave,nsz),dtype=float),np.zeros((nwave,nsz),dtype=float)
    pflux,epflux = np.zeros((nwave_p,nsz),dtype=float),np.zeros((nwave_p,nsz),dtype=float)

    fl,efl = np.zeros(nsz,dtype=float),np.zeros(nsz,dtype=float)
    i = -1
    for iy,ix in np.ndindex(np.shape(cube['flux'][0,:])):
        i+=1
        x[i],y[i] = xx[iy,ix],yy[iy,ix]
        flux[:,i],eflux[:,i] = cube['flux'][:,iy,ix],cube['fluxerr'][:,iy,ix]
        fl[i],efl[i] = np.nanmedian(cube['flux'][:,iy,ix]),np.nanmedian(cube['fluxerr'][:,iy,ix])
        pflux[:,i],epflux[:,i] = cube['photo_flux'][:,iy,ix],cube['photo_fluxerr'][:,iy,ix]
    ##invent my err to see if it works
   # efl=np.sqrt(fl) #np.sqrt(fl) #  0.1*fl #np.sqrt(fl)   ##

    ## Voronoi binning
    # xypixscale = (0.093,0.0895) - arcsec/pix
    ma = ((np.isfinite(efl)) & (efl > 0) & (fl > 0))
    voronoires = voronoi_2d_binning(x[ma],y[ma],fl[ma],efl[ma],snr,
                                    pixelsize=1.25,cvt=False,wvt=True,plot=True,quiet=0)
    binNum,xNode,yNode,xBar,yBar,sn,nPixels,scale = voronoires
    np.save(odir+gfile+'_voronoi'+str(snr)+'_2dbinning.npy',(binNum,xNode,yNode,xBar,yBar,sn,nPixels,scale,x[ma],y[ma]))


    ## Percentage selection
    rands = np.random.rand(len(xBar))
    rma = (rands < perc)

    ## Save file!
    #np.save(odir+gfile+'_voronoi'+str(snr)+'_2dbinning_output.npy',(binNum,xBar,yBar,ma,rma))
    fma = np.where(ma)[0]
    frma = np.where(rma)[0]

    # Re-organize into a new cube by:
    #  -rounding positions to closest x,y pixels
    #  -assign that position the sum of all the bin pixels and the rest to nan
    rxBar,ryBar = np.asarray(xBar[frma].round(),dtype=int),np.asarray(yBar[frma].round(),dtype=int)
    cube['flux'][:],cube['fluxerr'][:] = np.nan,np.nan
    cube['photo_flux'][:],cube['photo_fluxerr'][:] = np.nan,np.nan
    

    for b in range(0,len(rxBar)):
        #bma = (binNum == b)
        bma = (binNum == frma[b])
        fbma = np.where(bma)[0]
        weights = 1/eflux[:,fma[fbma]]**2
        weights_p = 1/epflux[:,fma[fbma]]**2

        mea = np.average(flux[:,fma[fbma]],axis=1,weights=weights)
        mea_p = np.average(pflux[:,fma[fbma]],axis=1,weights=weights_p)

        #mea = np.sum(flux[:,fma[fbma]],1)
        std = np.sqrt(np.average(((flux[:,fma[fbma]].T-mea)**2).T,axis=1,weights=weights))
        std_p = np.sqrt(np.average(((pflux[:,fma[fbma]].T-mea_p)**2).T,axis=1,weights=weights_p))

        er = np.sqrt(np.nansum(eflux[:,fma[fbma]]**2,axis=1))
        er_p = np.sqrt(np.nansum(epflux[:,fma[fbma]]**2,axis=1))
        cube['flux'][:,ryBar[b],rxBar[b]] = mea
        cube['fluxerr'][:,ryBar[b],rxBar[b]] = np.sqrt(er**2+std**2)#/1e20
        cube['photo_flux'][:,ryBar[b],rxBar[b]] = mea_p
        cube['photo_fluxerr'][:,ryBar[b],rxBar[b]] = np.sqrt(er_p**2+std_p**2)#/1e20
    #cube.writeto(home+'/crisp/AMUSING/ASASSN14co/_vor.fits')
    #print (cube['photo_flux'])
    return cube


# ---------------------
# Read MUSE/CALIFA IFU
# --------------------

def read_cube(ifu='muse',**kwargs):

    """ Read IFU data cube

    :file:
        The path+file of the cube
    """
    from astropy.io import fits
    ifufile = kwargs['ifufile']
    data = fits.open(ifufile)
    cube = {}
    if ifu == 'muse':
        cube['flux'] = data['spec_flux'].data
        cube['fluxerr'] = data['spec_fluxerr'].data
        cube['photo_flux'] = data['PHOTO_FLUX'].data
        cube['photo_fluxerr'] =data['PHOTO_FLUXERR'].data
        cube['wavelength'] = data['wave'].data
        cube['photo_region']=data['PHOTO_REGION'].data
        cube['spec_region']=data['SPEC_REGION'].data
        #wave
    	#CRVAL = float(header["CRVAL3"])
    	#NAXIS = int(header["NAXIS3"])
    	#CDELT = float(header["CD3_3"])
    	#CRPIX = float(header["CRPIX3"])
        #cube['wavelength'] = data['wave'].data
    elif ifu == 'califa':
    	cube['flux'] = data[0].data
    	cube['fluxerr'] = data[1].data
    	header = data[0].header
    	#CALIFA wave
    	CRVAL = float(header["CRVAL3"])
    	CDELT = float(header["CDELT3"])
    	NAXIS = int(header["NAXIS3"])
    	cube['wavelength'] = np.arange(NAXIS,dtype=float)*CDELT+CRVAL
    else:
    	print("Wrong IFU/data input")
    del data
    cube['mask'] = get_mask(cube['wavelength'],**kwargs)
    #cube['mask'] = get_mask1(cube['wavelength'],**kwargs)


    ## make voronoi binning if desired
    voronoi = kwargs['voronoi']
    if voronoi > -1:
        cube = voronoi_bin(cube,voronoi,**kwargs)

    return cube

# ---------------------
# Build IFU
# --------------------
def build_ifu(minvar=100,**kwargs):#cube=None

    x,y = kwargs['x'],kwargs['y']

    ## read cube
    if cube is None:
    	read_cube(**kwargs)

    from prospect.utils.obsutils import fix_obs

    ifu = {}
    #if dophot:
    galex = ['galex_FUV']
    spitzer = ['spitzer_irac_ch'+n for n in ['1','2','3','4']]
    hst = ['wfc3_uvis_f{0}'.format(b) for b in ['275w','555w','814w']]
    twomass= ['twomass_{0}'.format(b) for b in ['J','H','Ks']]
    wise = ['wise_w'+n for n in ['1']]

    filternames = galex+ hst + twomass + wise
    # And here we instantiate the `Filter()` objects using methods in `sedpy`,
    # and put the resultinf list of Filter objects in the "filters" key of the `obs` dictionary
    ifu["filters"] = sedpy.observate.load_filters(filternames)
    ifu["phot_wave"] = np.array([f.wave_effective for f in ifu["filters"]])
    #read_images(**kwargs) # need filterset
    ifu['maggies'] = (cube['photo_flux'][:,y,x]/1e17) *ifu['phot_wave']**2. * 3.34e4 /3631.0 # No photometry
    #ifu['filters'] = None # No photometry
    ifu["maggies_unc"] = (cube['photo_fluxerr'][:,y,x]/1e17) *ifu['phot_wave']**2. * 3.34e4 /3631.0#(1./20) * ifu["maggies"]
    # Converting from ergs/cm^2/s/Angs to maggies:
    # flux * wave**2. * 9.217
    ifu['wavelength'] = cube['wavelength']#/(1.0+kwargs['zred'])
    ifu['spectrum'] = (cube['flux'][:,y,x]/1e17)*cube['wavelength']**2. * 3.34e4 /3631.0
    #ifu['spectrum']=ifu['spectrum']/np.nanmedian(ifu['spectrum'])
    ifu['unc'] = (cube['fluxerr'][:,y,x]/1e17)*cube['wavelength']**2. * 3.34e4 /3631.0#*9.217*(1.0+kwargs['zred'])#2. * 3.34e4 /3631.0
    snr_profile = ifu['spectrum'] * ifu['unc']
    good= np.isfinite(snr_profile)
#     print (good)
#     print (ifu['spectrum'])
    ifu['spectrum']= ifu['spectrum'][good]

    ifu['unc']= ifu['unc'][good]
    ifu['wavelength']=ifu['wavelength'][good]
    #snr_profile = obs['spectrum'] * np.sqrt(obs['unc'])
    #good = np.isfinite(snr_profile)
    #cont, _ = fit_continuum(ifu["wavelength"], ifu["spectrum"], normorder=6, nreject=3)
    #cont = cont / cont.mean()
    #ifu["spectrum"] /= cont
    #ifu["unc"] /= cont    #ifu['spectrum']=ifu['spectrum']/np.nanmedian(ifu['spectrum'])

    #cube['mask'][1600:2500]=False
    ifu['mask'] = cube['mask'][good]
    ifu['success'] = True
    #ifu['phot_mask']= np.array(['wfc3_uvis_f275w' not in f.name for f in ifu["filters"]])
    # Make sure finite
    if (np.sum(np.isfinite(ifu['spectrum']*ifu['unc'])) < minvar):
        ifu['success'] = False
        return ifu

    # This ensures all required keys are present
    ifu = fix_obs(ifu)
    return ifu


# ----------------------
# Observational Mock Data
# ---------------------

def build_obs(add_noise=False, **kwargs):
    """Make a mock dataset.  Feel free to add more complicated kwargs, and put
    other things in the run_params dictionary to control how the mock is
    generated.

    :param snr:
        The S/N of the phock photometry.  This can also be a vector of same
        lngth as the number of filters.

    :param add_noise: (optional, boolean, default: True)
        If True, add a realization of the noise to the mock spectrum
    """
    from prospect.utils.obsutils import fix_obs

    filterset = kwargs['filterset'] ## works?
    snr_phot,snr_spec = kwargs['snr_phot'],kwargs['snr_spec']
    dophot,dospec = kwargs['phot'],(not kwargs['nospec'])

    # We'll put the mock data in this dictionary, just as we would for real data.
    # But we need to know the wavelengths if doing in which to generate mock data.
    mock = {}

    # Spectra (before calling model)
    if dospec:
        mock['wavelength'] = np.arange(3700.,7001.25,1.25)#*(1.0+zred)
        mock['filters'] = None # No photometry
    # Phot (before calling model)
    if dophot:
        mock['filters'] = load_filters(filterset)

    # We need the models to make a mock
    sps = build_sps(**kwargs)
    mod = build_model(**kwargs)

    # Now we get the mock params from the kwargs dict
    params = {}
    for p in mod.params.keys():
        if p in kwargs:
            params[p] = np.atleast_1d(kwargs[p])

    # And build the mock
    mod.params.update(params)
    spec, phot, _ = mod.mean_model(mod.theta, mock, sps=sps)

    # Now store some ancillary, helpful info;
    # this information is not required to run a fit.
    mock['true_spectrum'] = spec.copy()
    mock['true_maggies'] = phot.copy()
    mock['mock_params'] = deepcopy(mod.params)

    # --- Photometry
    if dophot:
        # And store the photometry, adding noise if desired
        pnoise_sigma = phot / snr_phot
        if add_noise:
            pnoise = np.random.normal(0, 1, len(phot)) * pnoise_sigma
            mock['maggies'] = phot + pnoise
        else:
            mock['maggies'] = phot.copy()
        mock['maggies_unc'] = pnoise_sigma
        mock['phot_mask'] = np.ones(len(phot), dtype=bool)
        ##Additional info
        mock['mock_snr_phot'] = snr_phot
        mock['phot_wave'] = np.array([f.wave_effective for f in load_filters(filterset)])
    else:
        mock['maggies'] = None # No photometry



    # --- Spectra
    # And store the spectrum, adding noise if desired
    if dospec:
        pnoise_sigma = spec / snr_spec
        if add_noise :
            pnoise = np.random.normal(0, 1, len(spec)) * pnoise_sigma
            mock['spectrum'] = (spec + pnoise)#/(1.0+zred)
        else :
            mock['spectrum'] = spec.copy()#/(1.0+zred)
        mock['unc'] = pnoise_sigma#/(1.0+zred)
        mock['mask'] = np.array(np.ones(mock['wavelength'].shape), bool)
        ##Additional info
        mock['mock_snr_spec'] = snr_spec
    else: # no spectrum
        mock['wavelength'] = None
        mock['spectrum'] = None

    mock['success'] = True

    # This ensures all required keys are present
    mock = fix_obs(mock)

    return mock

#--------------------
# Data object
#-------------------
def build_data(ifu='muse',**kwargs):#cube=None
    if (ifu == 'muse') or (ifu == 'califa'):
        return build_ifu(**kwargs)#cube=cube
    elif ifu == 'mock':
        return build_obs(**kwargs)
    else:
        print("Wrong IFU/data input")


# --------------
# SPS Object
# --------------

def build_sps(zcontinuous=1,**extras):
    """Instantiate and return the Stellar Population Synthesis object.

    :param zcontinuous: (default: 1)
        python-fsps parameter controlling how metallicity interpolation of the
        SSPs is acheived.  A value of `1` is recommended.
        * 0: use discrete indices (controlled by parameter "zmet")
        * 1: linearly interpolate in log Z/Z_\sun to the target metallicity
             (the parameter "logzsol".)
        * 2: convolve with a metallicity distribution function at each age.
             The MDF is controlled by the parameter "pmetals"
    """
    #from prospect.sources import CSPSpecBasis
    from prospect.sources import FastStepBasis
    #sps = CSPSpecBasis(zcontinuous=zcontinuous,flux_interp='linear',
     #                  compute_vega_mags=False)
    sps = FastStepBasis(zcontinuous=zcontinuous,flux_interp='linear',
                       compute_vega_mags=False)

    #sps.params['logzsol'] = -0.5
    #sps.params['gas_logz'] = -0.5
    #sps.params['gas_logu'] = -3.5
    return sps

# -----------------
# Noise Model
# ------------------

def build_noise(**extras):
    return None, None

# -----------------
# Get best fit
# ---------------------------
def get_best_fit(output, model, obs, sps):

   # model after optimization
   opt_results, topt = output["optimization"]
   ind_best = np.argmin([r.cost for r in opt_results])
   theta_best = opt_results[ind_best].x.copy()

   # error from optimization: std of all starting points
   if len(output) > 0:
       theta_all = np.asarray([r.x for r in opt_results])
       theta_error = np.std(theta_all,axis=0)
   else:
       theta_error = np.zeros(theta_best.shape)

   #best_spec, best_phot, _ = model.mean_model(theta_best, obs=obs, sps=sps)
   best_spec, best_phot, initial_mfrac = model.sed(theta_best, obs=obs, sps=sps)

   return theta_best, theta_error,best_spec,best_phot



# -----------
# Everything
# ------------

def build_all(**kwargs):

    return (build_data(**kwargs), build_model(**kwargs),
            build_sps(**kwargs), build_noise(**kwargs))


#------------
#  Loop spaxels: useful for multiprocessing
#---------------
def spaxel_fit(interval,runpars):#kwargs):

    print(interval)
    #print('interval' in locals())

    ## loop over spaxels
    for i in range(interval[0],interval[1]):
        for j in range(interval[2],interval[3]):
            if cube['spec_region'][j][i]==1 :
                rp = runpars.copy()
                if rp['docube']:
                    rp['x'] = i
                    rp['y'] = j
                    if rp['verbose']:
                        print("Doing spaxel %i and %i" %(i,j))
                add = '_'+str(rp['x'])+'-'+str(rp['y'])
    
                ## Get obs info
                obs = build_data(**rp)
    
                ## Set up an output file name, build fit ingredients, and run the fit
                hfile = rp['outdir']+rp['genfile']+add
                if not os.path.isfile(hfile+ 'add' +'.h5'):#'.npy'):
    
                    ## Prior on mass
                    mass = None
                    if rp['domassprior']:
                        mass = mass_prior(obs['wavelength'],obs['spectrum'],rp['zred'])
    
                    ## Get model (need to build from scratch cause of mass prior)
                    model = build_model(rp)
    
                    ## SPS
                    #sps = rp['sps']
                    if not obs['success']:
                        #writer.write_hdf5(hfile+'.h5', rp, model, obs,None,None)
                        #np.save(hfile+'.npy',(rp,model,obs,None))
                        continue
                    sps = build_sps(**rp)
                    #try:
                    output = fit_model(obs, model,sps, lnprob=lnprobfn, **rp)#sps in rp
    
                    if rp['verbose']:
                        print("Done optmization in {}s".format(output["optimization"][1]))
                        print(model.theta)
    
                    ## get best pars and spec to save
                    #bestpars,erpars,bestspec ,best_phot= get_best_fit(output,model,obs,sps)
    
                    ## Write results to output file
                    #write_h5file(hfile+'.h5',obs,spec=bestspec,pars=bestpars,erpars=erpars)
                    #np.warnings.filterwarnings('ignore', category=np.VisibleDeprecationWarning)
                    #np.save(hfile+'.npy',(rp,model,obs,output,bestpars,erpars,bestspec,best_phot))
                    #np.warnings.filterwarnings('ignore', category=np.VisibleDeprecationWarning)
                    #from prospect.io import write_results as writer
                    #run_params=rp.copy()
    
    
                    #hfile = "D:/IFS/demo_emcee_mcmc"
                    model=build_model(**rp)
                    writer.write_hdf5(hfile+'.h5', rp, model, obs,
                       output["sampling"][0], output["optimization"][0],
                       sps=sps,
                       tsample=output["sampling"][1],
                       toptimize=output["optimization"][1])
                    #np.save(hfile+'.npy',(rp, model, obs, output["sampling"][0], output["optimization"][0],tsample=output["sampling"][1],toptimize=output["optimization"][1]))
                    #except:
                     #   print(' skip ')
                else:
                    if rp['verbose']:
                        print("  Found file: %s" %(hfile+'.h5'))
                    #if rp['plot']:
                        #bestpars,erpars,bestspec = read_h5file(hfile+'.h5')
                        #run_params0,model,obs,output = np.load(hfile+'.npy',allow_pickle=True)
                        #rp,model,obs,output,bestpars,erpars,bestspec=np.load(hfile+'.npy',allow_pickle=True)
                        #np.warnings.filterwarnings('ignore', category=np.VisibleDeprecationWarning)
                        #import prospect.io.read_results as reader
                      #  results_type = "emcee" # | "dynesty"
    # grab results (dictionary), the obs dictionary, and our corresponding models
    # When using parameter files set `dangerous=True`
                        #results_type = "emcee"
                        #result, obs, _ = reader.results_from(hfile+'.h5'.format(results_type), dangerous=False)
    
    
                       # obs,rp = result[0]['obs'],result[0]['run_params']
    
                """if rp['plot'] and obs['success']:
                     #   plot_spectrum(hfile,obs,fitspec=None,photo=None,plotdir=rp['outdir'],## ONLY SPEC NOW!
                      #            plotfile=rp['genfile']+add+'_fitspec')
                        #
                       # plot_results(hfile,theta_best=None,plotdir=rp['outdir'],plotfile=rp['genfile']+add+'_corner')
    
                        #plot_results(hfile,theta_best=bestpars,plotdir=rp['outdir'],plotfile=rp['genfile']+add+'_corner')
                if 'obs' in locals(): del obs,rp
                if 'output' in locals(): del output"""
    return 1

ModuleNotFoundError: No module named 'mass_to_light_compute'

In [ ]:

# ---------------- MAIN -------------------------------
# ex (SN2009ds-NGC 3905): run CRISP_15d --x 278 --y 244 --ifu muse --zred 0.01926 --plot #308,167
# ASASSN-14co: zred=0.03334

if __name__=='__main__':

    # - Parser with default arguments -
    parser = prospect_args.get_parser()
    #home = os.path.expanduser('~')
    home="home/saby/coin/SN2006X_photfull/"
    # - Add custom arguments -


    ## General arguments
    parser.add_argument('--ifu', type=str,
                        default="muse",
                        help="IFU type: muse or califa or mock")
    parser.add_argument('--phot', action="store_true",
                        help="If set, add photometry")
    parser.add_argument('--nospec', action="store_true",
                        help="If set, don't do spectroscopy")

    ## IFU arguments
    parser.add_argument('--ifufile', type=str,
                        #default="/home/santiago/surveys/SELGIFS/observations/D-MA_2.V1200.cube.fits",
                        default=home+"corr_specphoto_fluxmap_SN2006X_reproject_woelli_unit.fits",
                        help="IFU path and file name.")
    parser.add_argument('--x', type=int,
                        default=-1,
                        help="IFU x position to do fit")
    parser.add_argument('--y', type=int,
                        default=-1,
                        help="IFU y position to do fit")
    parser.add_argument('--voronoi', type=int, default = 150,
                        help="IFU Voronoi binning S/N to do fit only to sum " )
    parser.add_argument('--perc', type=float, default = 1.0,
                        help="Percentage of pixels to use in fitting (randomly chosen) between 0-1 (def:1.0)" )

    ## Phot arguments
    parser.add_argument('--filterset', type=str, nargs="*",
                        default=['u-CSP', 'B-CSP'],#default=csp + dupont,
                        help="Names of filters through which to produce photometry.")
    parser.add_argument('--photname', type=str,
                        default='image',#default=csp + dupont,
                        help="Generic names of photometry image to add to filter name.")
    parser.add_argument('--maskfile', type=str, default='mask.dat',
                        help="Mask path and file name.")
    parser.add_argument('--nomask', action="store_true",
                        help="If set, no mask applied")


    ## Fit arguments
    parser.add_argument('--zred', type=float, default=0.005294, #0.024272
                        help="Redshift for the model (and mock).")
    parser.add_argument('--add_duste', action="store_true",
                        help="If set, add dust emission in the model (and mock).")
    parser.add_argument('--add_neb', action="store_true",
                        help="If set, add nebular emission in the model (and mock).")
    parser.add_argument('--add_noise', action="store_true",
                        help="If set, noise up the mock.")
    parser.add_argument('--snr_phot', type=float, default=20,
                        help="S/N ratio for the mock photometry.")
    parser.add_argument('--snr_spec', type=float, default=20,
                        help="S/N ratio for the mock spectra.")
    parser.add_argument('--tage', type=float, default=1.,
                        help="Age of the mock, Gyr.")
    parser.add_argument('--tau', type=float, default=1.,
                        help="SFH timescale parameter of the mock, Gyr.")
    parser.add_argument('--dust2', type=float, default=0.5,
                        help="Dust attenuation V band optical depth.")
    parser.add_argument('--logzsol', type=float, default=0.0,
                        help="Metallicity of the mock; log(Z/Z_sun).")
    parser.add_argument('--mass', type=float, default=1e5,
                        help="Stellar mass of the mock; solar masses formed.")
    parser.add_argument('--ssmooth', type=float, default=0.0,
                        help="Spectral smooth of the model (and mock) in angstrom (default MUSE).")
    parser.add_argument('--domassprior', action="store_true",
                        help="If set, do mass prior based on convolved I magnitude")
    parser.add_argument('--plot', action = 'store_true',
                        help='Perform statistic and plot trace and corner plots and perforf.')
    ## Parallel
    parser.add_argument('--multi', action = 'store_true',
                        help='Do multiple CPU for spaxels')
    ## Output arguments
    parser.add_argument('--outdir', type=str,default=home,
                        help='Output directory where to save all files')
    parser.add_argument('--genfile', type=str,default="SN2006X_phot_",
                        help='Generic output file ')

    ## Parse the supplied arguments, convert to a dictionary, and add this file for logging purposes
    args = parser.parse_args()
    run_params = vars(args)

    ## Build fit ingredients
    #obs, model, sps, noise = build_all(**run_params)
    sps = build_sps(**run_params)
    noise = build_noise(**run_params)
    model = build_model(**run_params)
    run_params["param_file"] = __file__
    run_params["sps_libraries"] = sps.ssp.libraries

    # - Configure parameters to run emcee fitting
    from prospect.fitting import lnprobfn
    run_params['nmin'] = 12
    run_params['optimize'] = True
    run_params['min_method'] = 'lm'
    run_params['emcee'] =False
   # run_params['init_disp'] =1.0
    
    run_params['dynesty'] = True
    run_params["verbose"] = True
    #run_params["sps"] = sps
    run_params["output_pickles"]=True
    run_params["add_duste"]= False
    run_params["add_neb"]= False
    run_params["zcontinuous"]= 1
    #run_params["imf_type"]= 1
    #run_params["sfh"]= 3
    run_params["nwalkers"] = 128
    run_params["niter"] = 1500
    run_params["nburn"] = [ 16,32,64]
    run_params["nested_rwalks"] = 40
    run_params['nested_weight_kwargs'] = {'pfrac': 1.0}
    run_params['nested_maxcall'] = 700000
    run_params['nested_maxcall_init'] = 700000
    run_params['nested_maxbatch'] = None
    run_params['nested_first_update'] = {'min_ncall': 20000, 'min_eff': 4.5}
    ## Full cube or single position
    cube = None
    interval = [0,1,0,1]
    run_params['docube'] = False
    if run_params['ifu'] != 'Muse':
        cube = read_cube(**run_params)
        if (run_params['x'] == -1) and (run_params['y'] == -1):
            nsp,ny,nx = cube['flux'].shape
            interval = [0,nx,0,ny]
            run_params['docube'] =True

            ## multi-processing
            if run_params['multi']:
                from multiprocessing import cpu_count,Pool
                from functools import partial
                ## get intervals
                num_processors = cpu_count()
                step = int(ny / num_processors)
                intervals = [[0,nx,i * step, i * step + step ] for i in range(num_processors)]
                if ((ny % step) != 0):   # add any "leftover" to an additional interval
                    intervals.append([0,nx,num_processors * step,ny] )

                ## parallel running
                pool = Pool(num_processors)
                result = pool.map(partial(spaxel_fit,runpars = run_params), intervals)

                pool.close()
                pool.join()
                #pdb.set_trace()
            else:
                spaxel_fit(interval,run_params),#**run_params
        else:
            spaxel_fit(interval,run_params)

    #mock
    else:
        build_obs(**run_params)
        spaxel_fit(interval,run_params)



   # run_params["sps_libraries"] = sps.ssp.libraries
   # run_params["param_file"] = __file__


#for i in range(2) :

#       if i==0 :
#           run_params['nmin'] = 32
#           run_params['nburn'] = [1]
#           run_params['niter'] = 3072
#       else :
#           run_params['nmin'] = 1
#           burn0, burn1 = get_burnin_time (output["sampling"][0])
#           print('\nComputed Burn-in times : %s, %s\n'%(burn0,burn1))
#           run_params['nburn'] = [burn0,burn1]
#           run_params['niter'] = 1024
#           model, init_spec = set_max (model, obs, sps)

#       hfile = "{0}_{1}_mcmc.h5".format(args.outfile, int(time.time()))
#       output = fit_model(obs, model, sps, noise, pool=Pool(2), **run_params)
#where get_burnin_time and set_max are two functions I wrote
#def get_burnin_time (sampler) :

#   aut_corr_time = np.max(sampler.get_autocorr_time(c=1))

#   return int(aut_corr_time*0.5), int(aut_corr_time)
#def set_max (mod, obs, sps) :
#   '''
#   Update the model to the MAP (of the first MCMC) as current theta vector
#   '''
#   mod.set_parameters(theta_max)
#
#   init_spec, init_phot, _ = mod.mean_model(mod.theta, obs, sps=sps)

#   return mod, init_spec
#the autocorrelation time comes from emcee package and computes the number of iterations needed for a MCMC to "forget" about the initial conditions/positions
#_mod.set_parameters(theta)_ is a *VERY* powerful Prospector method to re-initialize the model to a given theta vector.. imagine if you get a result from INLA, you write down a new vector with the values of the parameters in the right order and re-initialize the Prospector run
#Last but not least:
#If you need to speed up the optimization you can also pass _Pool(num)_ to _fit_model_ in order to use _num_ parallel processors at the same time
#But since it is a multithreading, sometimes the communication between many processors can get the things running slower (So, I am not using it on the server)
